# Hexapawn: Você vs. IA

Jogue contra a IA clicando nas peças azuis. A IA (vermelha) começa jogando aleatoriamente e aprende a cada partida.

> Este notebook usa widgets interativos do Google Colab. Para jogar, abra no Colab e execute todas as células (Ambiente de execução → Executar tudo).

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import random
import time
from google.colab import output


caixinhas = {}
historico_partida = []
vitorias_ia = 0
vitorias_voce = 0
log_vitorias = []
log_movimentos = []
contador_jogadas = 0

In [ ]:
############ mov ##############
def buscar_movimentos(tab, cor):
    lista = []
    direcao = 1 if cor == 'W' else -1
    inimigo = 'B' if cor == 'W' else 'W'

    for r in range(3):
        for c in range(3):

            if tab[r][c] == cor:

                # frente
                if 0 <= r + direcao < 3 and tab[r + direcao][c] == ' ':
                    lista.append(((r, c), (r + direcao, c)))

                # diagonal
                for dc in [-1, 1]:
                    if 0 <= r + direcao < 3 and 0 <= c + dc < 3:
                        if tab[r + direcao][c + dc] == inimigo:
                            lista.append(((r, c), (r + direcao, c + dc)))

    return lista

In [ ]:
########## ia #######################
def ia_escolhe_jogada(tab):
    global contador_jogadas
    estado = str(tab)
    if estado not in caixinhas:
        caixinhas[estado] = buscar_movimentos(tab, 'W')

    opcoes = caixinhas[estado]
    if not opcoes: return None

    escolha = random.choice(opcoes)
    historico_partida.append((estado, escolha))
    contador_jogadas += 1
    return escolha

def atualizar_aprendizado(ganhou):
    global vitorias_ia, vitorias_voce, historico_partida, contador_jogadas

    if historico_partida:

        if ganhou:
            # mais bolinhas
            for estado, mov in historico_partida:
                caixinhas[estado].append(mov)

        else:
            # tira ultima
            estado, mov = historico_partida[-1]
            if mov in caixinhas[estado]:
                caixinhas[estado].remove(mov)

    if ganhou: vitorias_ia += 1
    else: vitorias_voce += 1

    log_vitorias.append(1 if ganhou else 0)
    log_movimentos.append(contador_jogadas)
    historico_partida = []
    contador_jogadas = 0

In [ ]:
########### visual #################
tab_inicial = [['W','W','W'], [' ',' ',' '], ['B','B','B']]
tabuleiro = [linha[:] for linha in tab_inicial]
peca_selecionada = None
tela_saida = widgets.Output()

In [ ]:
############ graf ############
def mostrar_graficos():
    if not log_vitorias: return
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))

    fig.patch.set_facecolor('white')

    ax1.plot(range(1, len(log_movimentos)+1), log_movimentos, color='#7e8ac2', marker='o', markersize=6, linewidth=2)
    ax1.set_title("Movimentos caixinhas", fontsize=9)
    ax1.set_facecolor('white')
    ax1.set_xlabel("Partida", fontsize=8)
    ax1.grid(True, linestyle=':', alpha=0.6)

    ax2.bar(['Você', 'IA'], [vitorias_voce, vitorias_ia], color='#7e8ac2', edgecolor='#7e8ac2')
    ax2.set_title(f"Placar: {vitorias_voce} x {vitorias_ia}", fontsize=9)
    ax2.set_facecolor('white')

    plt.tight_layout()
    plt.show()

In [ ]:
############ historico ############
def mostrar_historico():
    if not log_vitorias: return
    linhas = ""
    for i, v in enumerate(log_vitorias):
        vencedor = "IA venceu" if v == 1 else "Você venceu"
        fundo    = "#fff0f0" if v == 1 else "#f0f4ff"
        linhas  += f"""
            <tr style="background:{fundo};">
                <td style="padding:4px 12px; color:#555;">Partida {i+1}</td>
                <td style="padding:4px 12px; font-weight:bold;">{vencedor}</td>
                <td style="padding:4px 12px; color:#888;">{log_movimentos[i]} jogadas</td>
            </tr>
        """
    display(widgets.HTML(f"""
        <div style="max-height:180px; overflow-y:auto; margin-top:10px;
                    border:1px solid #ddd; border-radius:6px; font-family:sans-serif; font-size:13px;">
           <table style="width:100%; border-collapse:collapse; color:#000;">
                <thead>
                    <tr style="background:#f5f5f5; color:#333;">
                        <th style="padding:5px 12px; text-align:left;">Partida</th>
                        <th style="padding:5px 12px; text-align:left;">Resultado</th>
                        <th style="padding:5px 12px; text-align:left;">Jogadas</th>
                    </tr>
                </thead>
                <tbody>{linhas}</tbody>
            </table>
        </div>
    """))

In [ ]:
############ jogo ################

def desenhar_jogo(msg="Sua vez!", fim=False):
    with tela_saida:
        clear_output(wait=True)
        if fim:
            display(widgets.HTML(f"""
                <div style="background-color: black; color: white; width: 300px; height: 300px;
                            display: flex; flex-direction: column; justify-content: center;
                            align-items: center; border-radius: 8px; text-align: center; font-family: sans-serif;">
                    <h2 style="margin: 0;">FIM</h2>
                    <p style="padding: 15px; font-weight: bold;">{msg}</p>
                    <button onclick="google.colab.kernel.invokeFunction('reiniciar', [], {{}})"
                            style="padding: 8px 15px; font-weight: bold; cursor: pointer; background: white; border: none; border-radius: 4px;">RECOMEÇAR</button>
                </div>
            """))
        else:
            print(f"PARTIDA {len(log_vitorias) + 1} — {msg}")
            grade = widgets.GridspecLayout(3, 3, width='300px', height='300px')
            for r in range(3):
                for c in range(3):
                    p = tabuleiro[r][c]
                    cor = '#EEEEEE' if (r+c)%2==0 else '#CCCCCC'
                    if p == 'B': cor = '#0055FF'
                    elif p == 'W': cor = '#FF3333'

                    b = widgets.Button(description=p, layout=widgets.Layout(height='95px', width='95px'))
                    b.style.button_color = cor
                    b.style.text_color = 'white'
                    b.style.font_weight = 'bold'
                    b.on_click(lambda x, row=r, col=c: clique_do_usuario(row, col))
                    grade[r, c] = b
            display(grade)
        mostrar_graficos()
        mostrar_historico()

def clique_do_usuario(r, c):
    global peca_selecionada, tabuleiro
    if peca_selecionada is None:
        if tabuleiro[r][c] == 'B': peca_selecionada = (r, c)
    else:
        sr, sc = peca_selecionada
        if ((sr, sc), (r, c)) in buscar_movimentos(tabuleiro, 'B'):
            tabuleiro[sr][sc] = ' '
            tabuleiro[r][c] = 'B'
            peca_selecionada = None
            if r == 0 or not buscar_movimentos(tabuleiro, 'W'): terminar(False)
            else: turno_ia()
        else:
            peca_selecionada = None
            desenhar_jogo("Movimento inválido!")

def turno_ia():
    time.sleep(0.4)
    m = ia_escolhe_jogada(tabuleiro)
    if not m:
        terminar(False)
        return
    (r1, c1), (r2, c2) = m
    tabuleiro[r1][c1] = ' '
    tabuleiro[r2][c2] = 'W'
    desenhar_jogo("IA moveu")
    time.sleep(0.6)
    if r2 == 2 or not buscar_movimentos(tabuleiro, 'B'): terminar(True)
    else: desenhar_jogo()

def terminar(ia_ganhou):
    global tabuleiro
    atualizar_aprendizado(ia_ganhou)
    txt = "A IA venceu!" if ia_ganhou else "Você venceu!"
    tabuleiro = [linha[:] for linha in tab_inicial]
    desenhar_jogo(txt, fim=True)


###############################################

output.register_callback('reiniciar', lambda: desenhar_jogo())
display(tela_saida)
desenhar_jogo()